# A. Chargement des données

Chargement des données dans un dataframe.

Dans ce notebook on utilisera le dataset fournis par Databricks `nyctaxi`

In [0]:
from pyspark.sql.functions import *

voyages_df = spark.read.table("samples.nyctaxi.trips")

display(voyages_df.limit(10))

# B. Opérations de regroupement

## Fonctions de regroupement avec Spark

Les fonctions de regroupement permettent d'agréger des données selon une ou plusieurs colonnes. Avec PySpark, on utilise principalement `groupBy` suivi de fonctions d'agrégation comme `count`, `avg`, `sum`, etc.

**Exemples :**

- Compter le nombre de trajets par zone de départ :
  ```python
  voyages_df.groupBy("pickup_zip").count()
  ```
  

- Calculer la distance moyenne et le montant total des courses par zone de départ :
  ```python
  voyages_df.groupBy("pickup_zip").agg(
      avg("trip_distance"),
      sum("fare_amount")
  )
  ```
  

- Grouper par plusieurs colonnes :
  ```python
  voyages_df.groupBy("pickup_zip", "dropoff_zip").count()
  ```
  

Ces opérations sont utiles pour analyser et résumer de grandes quantités de données.

In [0]:
# regrouper les données par zipcode de départ pour déterminer la zone de départ la plus utilisée
location_counts = voyages_df \
    .groupBy("pickup_zip") \
    .count() \
    .orderBy(desc("count"))

display(location_counts.limit(10))

# C. Multiple opérations de groupement

L'agrégation et le regroupement avec plusieurs fonctions permettent d'obtenir des statistiques variées pour chaque groupe de données. Par exemple, en regroupant par une colonne comme `pickup_zip`, on peut calculer simultanément le nombre de trajets, la distance moyenne, le montant moyen et le montant total des courses. Cela se fait grâce à la méthode `agg` de PySpark, qui accepte plusieurs fonctions d'agrégation dans une seule opération. Cette technique est idéale pour analyser rapidement et efficacement les caractéristiques principales de chaque groupe.

L'agrégation permet de résumer et d'analyser des ensembles de données en appliquant des fonctions statistiques (comme la somme, la moyenne, le minimum, le maximum, etc.) sur des groupes de lignes partageant une ou plusieurs valeurs communes.

Avec PySpark, on utilise `groupBy` pour regrouper les données selon une ou plusieurs colonnes, puis on applique plusieurs fonctions d'agrégation grâce à la méthode `agg`. Cela permet de calculer plusieurs statistiques en une seule opération, ce qui est efficace et lisible.

**Exemple :**

Pour chaque zone de départ (`pickup_zip`), on peut calculer :
* Le nombre total de trajets
* La distance moyenne parcourue
* Le montant moyen et total des courses

Cela se fait en combinant plusieurs fonctions dans `agg`, comme illustré dans la cellule suivante. Cette approche est très puissante pour obtenir rapidement un résumé statistique par groupe.

In [0]:
location_stats = voyages_df \
    .groupBy("pickup_zip") \
    .agg(
        count("*").alias("total_trips"),
        round(avg("trip_distance"), 2).alias("avg_trip_distance"),
        round(avg("fare_amount"), 2).alias("avg_fare_amount"),
        round(sum("fare_amount"), 2).alias("total_fare_amount")
    ) \
    .orderBy(desc("total_trips"))

display(location_stats.limit(10))

# D. Fonctions de fenêtrage

## Fonctions de fenêtrage avec PySpark

Les fonctions de fenêtrage (window functions) permettent d'effectuer des calculs sur des groupes de lignes liés à la ligne courante, sans regrouper les données comme avec `groupBy`. Elles sont utiles pour calculer des statistiques mobiles, des rangs, des cumuls, etc.

Avec PySpark, on utilise le module `pyspark.sql.window` pour définir une fenêtre sur laquelle appliquer des fonctions comme `row_number`, `rank`, `sum`, `avg`, etc.

**Exemple d'utilisation :**

```python
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, sum

# Définir une fenêtre partitionnée par 'pickup_zip' et ordonnée par 'fare_amount' décroissant
window_spec = Window.partitionBy("pickup_zip").orderBy(desc("fare_amount"))

# Ajouter le rang de chaque course dans sa zone de départ
voyages_df.withColumn("rank", row_number().over(window_spec))

# Calculer le cumul du montant des courses par zone de départ, ordonné par date
window_cum = Window.partitionBy("pickup_zip").orderBy("pickup_datetime").rowsBetween(Window.unboundedPreceding, Window.currentRow)
voyages_df.withColumn("cumulative_fare", sum("fare_amount").over(window_cum))
```

Les fonctions de fenêtrage sont puissantes pour analyser des tendances, des évolutions ou des classements au sein de groupes de données.

In [0]:
from pyspark.sql.window import Window

window_by_trips = Window.orderBy(desc("total_trips"))
window_by_fare = Window.orderBy(desc("avg_fare_amount"))

ranked_locations = location_stats \
    .withColumn("rank_by_trips", row_number().over(window_by_trips)) \
    .withColumn("rank_by_fare", row_number().over(window_by_fare)) \
    .withColumn("fare_quintile", ntile(5).over(window_by_fare)) # divise en 5 groupe par tarif (fare)

In [0]:
display(ranked_locations.select(
    "pickup_zip",
    "total_trips",
    "avg_fare_amount",
    "avg_trip_distance",
    "rank_by_trips",
    "rank_by_fare",
    "fare_quintile"
).limit(15))